In [2]:
pip install spacy

   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
    --------------------------------------- 0.3/14.3 MB ? eta -:--:--
    --------------------------------------- 0.3/14.3 MB ? eta -:--:--
    --------------------------------------- 0.3/14.3 MB ? eta -:--:--
    --------------------------------------- 0.3/14.3 MB ? eta -:--:--
    --------------------------------------- 0.3/14.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/14.3 MB 231.6 kB/s eta 0:01:00
   - -------------------------------------- 0.5/14.3 MB 231.6 kB/s eta 0:01:00
   - -------------------------------------- 0.5/14.3 MB 231.6 kB/s eta 0

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

In [9]:
import pandas as pd

df = pd.read_csv(r'D:\vs code projects\yt_comment_analyzer\data\processed\final_data.csv')
df.shape

(199508, 2)

In [10]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

In [11]:
X = df['clean_comment']
y = df['category']

In [16]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 769.9 kB/s eta 0:00:16
     -- ------------------------------------- 0.8/12.8 MB 1.1 MB/s eta 0:00:11
     ---- ----------------------------------- 1.3/12.8 MB 1.3 MB/s eta 0:00:09
     ---- ----------------------------------- 1.6/12.8 MB 1.4 MB/s eta 0:00:09
     ------ --------------------------------- 2.1/12.8 MB 1.5 MB/s eta 0:00:08
     ------- -------------------------------- 2.4/12.8 MB 1.6 MB/s eta 0:00:07
     -------- ------------------------------- 2.6/12.8 MB 1.5 MB/s eta 0:00:07
     --------- ------------------------------ 3.1/12.8 MB 1.5 MB/s eta 0:00:07
     ---------- ----------------------------- 3.4/12.8 MB 1.6 MB/s eta 0:00:07
     ----------- ---------------------------- 3.7/12.8 MB 1.5 MB/s eta 0

In [17]:
# Load spacy language model for POS tagging
nlp = spacy.load('en_core_web_sm')

In [18]:
# Function to extract custom features
def extract_custom_features(text):
    doc = nlp(text)
    word_list = [token.text for token in doc]

    # 1. Comment Length (number of characters)
    comment_length = len(text)

    # 2. Word Count
    word_count = len(word_list)

    # 3. Average Word Length
    avg_word_length = sum(len(word) for word in word_list) / word_count if word_count > 0 else 0

    # 4. Unique Word Count
    unique_word_count = len(set(word_list))

    # 5. Lexical Diversity
    lexical_diversity = unique_word_count / word_count if word_count > 0 else 0

    # 6. Count of POS Tags
    pos_count = len([token.pos_ for token in doc])

    # 7. Proportion of POS Tags
    pos_tags = [token.pos_ for token in doc]
    pos_proportion = {tag: pos_tags.count(tag) / word_count for tag in set(pos_tags)} if word_count > 0 else {}

    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'unique_word_count': unique_word_count,
        'lexical_diversity': lexical_diversity,
        'pos_count': pos_count,
        **pos_proportion  # Flattening the POS proportions
    }

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [20]:
# Apply the custom feature extraction
train_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_train])
test_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_test])

In [21]:
train_custom_features.head()

,comment_length,word_count,avg_word_length,unique_word_count,lexical_diversity,pos_count,VERB,NOUN,ADJ,AUX,...,ADP,ADV,PART,CCONJ,INTJ,DET,SCONJ,X,PUNCT,SYM
0,46,7,5.714286,7,1.000000,7,0.714286,0.285714,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,441,58,6.620690,4,0.068966,58,NaN,0.327586,0.672414,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,72,10,6.300000,9,0.900000,10,0.100000,0.400000,0.100000,0.100000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,146,24,5.166667,24,1.000000,24,0.291667,0.291667,0.083333,0.041667,...,0.041667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,157,22,6.181818,22,1.000000,22,0.363636,0.318182,0.181818,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
# Replace NaN values in POS tag proportions with 0
train_custom_features.fillna(0, inplace=True)
test_custom_features.fillna(0, inplace=True)

,comment_length,word_count,avg_word_length,unique_word_count,lexical_diversity,pos_count,PROPN,VERB,NOUN,PRON,...,ADP,PART,CCONJ,NUM,SCONJ,DET,X,INTJ,PUNCT,SYM
0,57,9,5.444444,9,1.000000,9,0.111111,0.222222,0.555556,0.111111,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
1,36,5,6.400000,5,1.000000,5,0.200000,0.200000,0.600000,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
2,33,5,5.800000,5,1.000000,5,0.200000,0.200000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,184,24,6.708333,22,0.916667,24,0.000000,0.250000,0.583333,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,40,6,5.833333,6,1.000000,6,0.000000,0.500000,0.000000,0.166667,...,0.166667,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39897,48,7,6.000000,7,1.000000,7,0.000000,0.142857,0.571429,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
39898,30,4,6.750000,4,1.000000,4,0.250000,0.250000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
39899,43,7,5.285714,7,1.000000,7,0.142857,0.142857,0.428571,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
39900,122,18,5.833333,18,1.000000,18,0.111111,0.277778,0.333333,0.000000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.111111,0.0,0.0


In [23]:
test_custom_features.isnull().sum()

comment_length       0
word_count           0
avg_word_length      0
unique_word_count    0
lexical_diversity    0
pos_count            0
PROPN                0
VERB                 0
NOUN                 0
PRON                 0
AUX                  0
ADJ                  0
ADV                  0
ADP                  0
PART                 0
CCONJ                0
NUM                  0
SCONJ                0
DET                  0
X                    0
INTJ                 0
PUNCT                0
SYM                  0
dtype: int64

In [24]:
# Apply TfidfVectorizer with trigram setting and max_features=1000
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=1000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [25]:
# Convert TF-IDF to DataFrame
X_train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf.get_feature_names_out())
X_test_tfidf_df = pd.DataFrame(X_test_tfidf.toarray(), columns=tfidf.get_feature_names_out())

In [26]:
# Combine TF-IDF and custom features
X_train_combined = pd.concat([X_train_tfidf_df.reset_index(drop=True), train_custom_features.reset_index(drop=True)], axis=1)
X_test_combined = pd.concat([X_test_tfidf_df.reset_index(drop=True), test_custom_features.reset_index(drop=True)], axis=1)

In [27]:
X_train_combined

,100,2012,2014,2019,4th,72000,aap,able,absolutely,abt,...,ADP,ADV,PART,CCONJ,INTJ,DET,SCONJ,X,PUNCT,SYM
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.041667,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.185185,0.000000,0.074074,0.0,0.037037,0.0,0.0,0.0,0.0
159602,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.066667,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
159603,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.083333,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0
159604,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.142857,0.0,0.142857,0.0,0.0,0.0,0.0


In [28]:
from lightgbm import LGBMClassifier

In [29]:
model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    is_unbalance=True,
    class_weight="balanced",
    reg_alpha=0.01215570124339254,  # L1 regularization
    reg_lambda=0.07146814041554751,  # L2 regularization,
    learning_rate=0.03721005633871337,
    n_estimators=997,
    max_depth=9,
    num_leaves=133,
    min_child_samples=31,
    colsample_bytree=0.9234118017408489,
    subsample=0.8740422770140972
)

In [30]:
# Fit the model on the resampled training data
model.fit(X_train_combined, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.258798 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 225705
[LightGBM] [Info] Number of data points in the train set: 159606, number of used features: 1022
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

,num_leaves,133
,max_depth,9
,learning_rate,0.03721005633871337
,n_estimators,997
,objective,'multiclass'
,class_weight,'balanced'
,min_child_samples,31
,subsample,0.8740422770140972
,colsample_bytree,0.9234118017408489
,reg_alpha,0.01215570124339254
,reg_lambda,0.07146814041554751


In [31]:
# Predict on the test set
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test_combined)
accuracy = accuracy_score(y_test, y_pred)
accuracy

0.7103403338178538

In [32]:
from sklearn.metrics import classification_report
# Generate classification report
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.79      0.65      0.72     13554
           1       0.89      0.73      0.80     17597
           2       0.47      0.76      0.58      8751

    accuracy                           0.71     39902
   macro avg       0.72      0.71      0.70     39902
weighted avg       0.76      0.71      0.72     39902

